# 🏎️ F1 Driver–Circuit Compatibility
## Notebook 04 — K-Means Clustering

**What this notebook does:**
1. Standardizes features (zero mean, unit variance)
2. Finds optimal K using Elbow + Silhouette methods
3. Fits K-Means separately for drivers (K=4) and circuits (K=3) — **training data only**
4. Assigns cluster labels to ALL rows including 2025 test set
5. Names and interprets each cluster
6. Saves `final_dataset.csv` ready for modeling

**Input:**  `data/processed/driver_features.csv`, `data/processed/circuit_features.csv`  
**Output:** `data/processed/final_dataset.csv`

⚠️ **Leakage prevention rules:**
- K-Means is fit on 2019–2024 only; 2025 rows are assigned via `predict()`
- `teammate_pace_delta` and `position_gain` are never passed to K-Means
- `StandardScaler` is fit on training data only

In [ ]:
%pip install scikit-learn --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster       import KMeans
from sklearn.metrics       import silhouette_score
from sklearn.decomposition import PCA

RANDOM_STATE = 42
print('✅ Imports ready.')

## Step 1 — Configure Paths

In [ ]:
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
FIGURES_DIR  = os.path.join(PROJECT_ROOT, 'figures')
MODELS_DIR   = os.path.join(PROJECT_ROOT, 'models')

DRIVER_FEAT_PATH  = os.path.join(PROC_DIR, 'driver_features.csv')
CIRCUIT_FEAT_PATH = os.path.join(PROC_DIR, 'circuit_features.csv')
FINAL_DATA_PATH   = os.path.join(PROC_DIR, 'final_dataset.csv')

for p in [DRIVER_FEAT_PATH, CIRCUIT_FEAT_PATH]:
    print(f"  {'✅' if os.path.exists(p) else '❌ MISSING'} {os.path.basename(p)}")

## Step 2 — Load Features

In [ ]:
driver_features  = pd.read_csv(DRIVER_FEAT_PATH)
circuit_features = pd.read_csv(CIRCUIT_FEAT_PATH)

# Standardize circuit names
NAME_MAP = {'Monte Carlo': 'Monaco', 'Sao Paulo': 'São Paulo', 'Mexico': 'Mexico City'}
driver_features['circuit_id']  = driver_features['circuit_id'].replace(NAME_MAP)
circuit_features['circuit_id'] = circuit_features['circuit_id'].replace(NAME_MAP)

# Re-aggregate circuits that now share same name after remapping
circ_agg_cols = circuit_features.select_dtypes(include='number').columns.tolist()
circuit_features = circuit_features.groupby('circuit_id')[circ_agg_cols].mean().reset_index()

TRAIN_SEASONS = [2019, 2020, 2021, 2022, 2023, 2024]
TEST_SEASONS  = [2025]

driver_train = driver_features[driver_features['season'].isin(TRAIN_SEASONS)].copy()
driver_test  = driver_features[driver_features['season'].isin(TEST_SEASONS)].copy()

print(f'✅ driver_features: {driver_features.shape} | train: {len(driver_train)} | test: {len(driver_test)}')
print(f'✅ circuit_features: {circuit_features.shape}')

## Step 3 — Define Feature Sets

In [ ]:
DRIVER_CLUSTER_COLS = [
    'avg_lap_time_norm', 'lap_time_std', 'best_lap_time_norm',
    'avg_sector1_norm', 'avg_sector2_norm', 'avg_sector3_norm',
    'tyre_degradation_slope', 'avg_stint_length', 'number_of_pit_stops',
]

CIRCUIT_CLUSTER_COLS = [
    'lap_time_variability', 'sector1_dominance', 'sector2_dominance', 'sector3_dominance',
    'avg_pit_stops_per_race', 'avg_stint_length', 'tyre_degradation_avg',
    'overtaking_index', 'dnf_rate',
]

for col in DRIVER_CLUSTER_COLS:
    assert col in driver_features.columns, f'MISSING driver feature: {col}'
for col in CIRCUIT_CLUSTER_COLS:
    assert col in circuit_features.columns, f'MISSING circuit feature: {col}'

print(f'✅ Driver clustering features:  {len(DRIVER_CLUSTER_COLS)}')
print(f'✅ Circuit clustering features: {len(CIRCUIT_CLUSTER_COLS)}')

## Step 4 — Standardize Features

In [ ]:
driver_scaler = StandardScaler()

X_driver_train = driver_train[DRIVER_CLUSTER_COLS].dropna()
X_driver_all   = driver_features[DRIVER_CLUSTER_COLS].dropna()

driver_scaler.fit(X_driver_train)  # Fit on TRAIN ONLY

X_driver_train_scaled = driver_scaler.transform(X_driver_train)
X_driver_all_scaled   = driver_scaler.transform(X_driver_all)

circuit_scaler = StandardScaler()
X_circuit = circuit_features[CIRCUIT_CLUSTER_COLS].dropna()
X_circuit_scaled = circuit_scaler.fit_transform(X_circuit)

print(f'✅ Driver train matrix:  {X_driver_train_scaled.shape}')
print(f'✅ Driver full matrix:   {X_driver_all_scaled.shape}')
print(f'✅ Circuit matrix:       {X_circuit_scaled.shape}')

## Step 5 — Find Optimal K: Driver Clusters

In [ ]:
K_RANGE = range(2, 10)
driver_inertias, driver_silhouettes = [], []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_driver_train_scaled)
    driver_inertias.append(km.inertia_)
    driver_silhouettes.append(silhouette_score(X_driver_train_scaled, labels))
    print(f'  K={k}  inertia={km.inertia_:,.0f}  silhouette={driver_silhouettes[-1]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(list(K_RANGE), driver_inertias, 'o-', color='#e10600', linewidth=2)
axes[0].set_title('Driver Clusters — Elbow Method', fontweight='bold')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia'); axes[0].grid(alpha=0.3)

axes[1].plot(list(K_RANGE), driver_silhouettes, 's-', color='#1f77b4', linewidth=2)
axes[1].set_title('Driver Clusters — Silhouette Score', fontweight='bold')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'driver_cluster_selection.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'\n💡 Mathematical max silhouette at K={list(K_RANGE)[driver_silhouettes.index(max(driver_silhouettes))]}')
print('   We choose K=4 to capture behaviorally meaningful archetypes (Frontrunner/Aggressive/Tyre Manager/Backmarker)')

## Step 6 — Find Optimal K: Circuit Clusters

In [ ]:
K_RANGE_CIRC = range(2, 8)
circuit_inertias, circuit_silhouettes = [], []

for k in K_RANGE_CIRC:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels = km.fit_predict(X_circuit_scaled)
    circuit_inertias.append(km.inertia_)
    circuit_silhouettes.append(silhouette_score(X_circuit_scaled, labels))
    print(f'  K={k}  inertia={km.inertia_:,.2f}  silhouette={circuit_silhouettes[-1]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(list(K_RANGE_CIRC), circuit_inertias, 'o-', color='#e10600', linewidth=2)
axes[0].set_title('Circuit Clusters — Elbow Method', fontweight='bold')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia'); axes[0].grid(alpha=0.3)

axes[1].plot(list(K_RANGE_CIRC), circuit_silhouettes, 's-', color='#1f77b4', linewidth=2)
axes[1].set_title('Circuit Clusters — Silhouette Score', fontweight='bold')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'circuit_cluster_selection.png'), dpi=150, bbox_inches='tight')
plt.show()

## Step 7 — Fit Final K-Means Models

In [ ]:
K_DRIVERS  = 4
K_CIRCUITS = 3

# ── Driver clustering ─────────────────────────────────────────────────────
km_drivers = KMeans(n_clusters=K_DRIVERS, random_state=RANDOM_STATE, n_init=20)
km_drivers.fit(X_driver_train_scaled)  # Fit on TRAIN only

driver_all_idx = driver_features[DRIVER_CLUSTER_COLS].dropna().index
driver_features.loc[driver_all_idx, 'driver_cluster'] = km_drivers.predict(X_driver_all_scaled)
driver_features['driver_cluster'] = driver_features['driver_cluster'].astype('Int64')

print('✅ Driver clusters assigned:')
print(driver_features['driver_cluster'].value_counts().sort_index().to_string())

# ── Circuit clustering ────────────────────────────────────────────────────
km_circuits = KMeans(n_clusters=K_CIRCUITS, random_state=RANDOM_STATE, n_init=20)
km_circuits.fit(X_circuit_scaled)  # Fit on all circuits (all from training seasons)

circ_idx = circuit_features[CIRCUIT_CLUSTER_COLS].dropna().index
circuit_features.loc[circ_idx, 'circuit_cluster'] = km_circuits.predict(X_circuit_scaled)
circuit_features['circuit_cluster'] = circuit_features['circuit_cluster'].astype('Int64')

print('\n✅ Circuit clusters assigned:')
print(circuit_features[['circuit_id', 'circuit_cluster']].sort_values('circuit_cluster').to_string(index=False))

## Step 8 — Name Clusters

Inspect centroids and assign interpretable names.

In [ ]:
# Driver centroid inspection
centroid_df = pd.DataFrame(
    driver_scaler.inverse_transform(km_drivers.cluster_centers_),
    columns=DRIVER_CLUSTER_COLS
)
centroid_df.index.name = 'cluster'
print('Driver cluster centroids (original scale):')
print(centroid_df[['avg_lap_time_norm', 'lap_time_std', 'avg_stint_length', 'number_of_pit_stops']].round(3).to_string())
print('\n📝 Assign names below based on these centroid values.')
print('   Fastest pace (lowest avg_lap_time_norm) = Frontrunner')
print('   Most pit stops, short stints = Aggressive')
print('   Fewest pit stops, long stints = Tyre Manager')
print('   Slowest + most inconsistent = Backmarker')

In [ ]:
# ── UPDATE THESE based on your centroid printout above ────────────────────
# Map cluster integer → name (adjust if your cluster numbers differ)
DRIVER_CLUSTER_NAMES = {
    0: 'Aggressive',
    1: 'Frontrunner',
    2: 'Tyre Manager',
    3: 'Backmarker',
}

# Circuit names — inspect circuit_features printout above
# Monza/Spa/Silverstone/Suzuka → High-Speed
# Monaco/Istanbul/Jeddah → High-Degradation
# Hockenheim (2019 only) → Outlier
CIRCUIT_CLUSTER_NAMES = {
    0: 'High-Speed',
    1: 'High-Degradation',
    2: 'Outlier',
}

driver_features['driver_cluster_name']  = driver_features['driver_cluster'].map(DRIVER_CLUSTER_NAMES)
circuit_features['circuit_cluster_name'] = circuit_features['circuit_cluster'].map(CIRCUIT_CLUSTER_NAMES)

print('✅ Cluster names applied.')
print('\nDriver cluster distribution:')
print(driver_features['driver_cluster_name'].value_counts().to_string())

## Step 9 — PCA Visualization

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_driver_train_scaled)

train_labels = km_drivers.predict(X_driver_train_scaled)

# Map test points
X_driver_test_clean = driver_test[DRIVER_CLUSTER_COLS].dropna()
X_driver_test_scaled = driver_scaler.transform(X_driver_test_clean)
X_pca_test = pca.transform(X_driver_test_scaled)

COLOR_MAP = {0: '#e10600', 1: '#1f77b4', 2: '#2ca02c', 3: '#ff7f0e'}
NAMES_MAP  = {v: k for k, v in DRIVER_CLUSTER_NAMES.items()}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Driver PCA
ax = axes[0]
for cluster_id, name in DRIVER_CLUSTER_NAMES.items():
    mask = train_labels == cluster_id
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=COLOR_MAP[cluster_id], label=f'Cluster {cluster_id}: {name}',
               alpha=0.4, s=10, edgecolors='none')

ax.scatter(X_pca_test[:, 0], X_pca_test[:, 1],
           c='black', marker='*', s=60, label='2025 test', zorder=5)
ax.set_title(f'Driver Clusters — PCA Projection\n(PC1+PC2 explain {pca.explained_variance_ratio_.sum()*100:.1f}% of variance)',
             fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend(fontsize=8, markerscale=2)
ax.grid(alpha=0.2)

# Circuit PCA
pca_circ = PCA(n_components=2, random_state=RANDOM_STATE)
X_circ_pca = pca_circ.fit_transform(X_circuit_scaled)
circ_labels = km_circuits.predict(X_circuit_scaled)
circ_names_clean = circuit_features[CIRCUIT_CLUSTER_COLS].dropna().index

ax2 = axes[1]
CIRC_COLORS = {0: '#e10600', 1: '#1f77b4', 2: '#2ca02c'}
for cluster_id, name in CIRCUIT_CLUSTER_NAMES.items():
    mask = circ_labels == cluster_id
    ax2.scatter(X_circ_pca[mask, 0], X_circ_pca[mask, 1],
                c=CIRC_COLORS[cluster_id], label=f'{name}', s=60, edgecolors='none')

# Label each circuit
for i, circ in enumerate(circuit_features.loc[circuit_features[CIRCUIT_CLUSTER_COLS].dropna().index, 'circuit_id']):
    ax2.annotate(circ, (X_circ_pca[i, 0], X_circ_pca[i, 1]), fontsize=6, alpha=0.7)

ax2.set_title(f'Circuit Clusters — PCA Projection\n({pca_circ.explained_variance_ratio_.sum()*100:.1f}% variance explained)',
              fontweight='bold')
ax2.set_xlabel(f'PC1 ({pca_circ.explained_variance_ratio_[0]*100:.1f}% variance)')
ax2.set_ylabel(f'PC2 ({pca_circ.explained_variance_ratio_[1]*100:.1f}% variance)')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'cluster_pca.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: figures/cluster_pca.png')

## Step 10 — Assemble Final Dataset

In [ ]:
# Merge circuit cluster labels into driver features
circ_label_map = circuit_features[['circuit_id', 'circuit_cluster', 'circuit_cluster_name']].copy()
final_df = driver_features.merge(circ_label_map, on='circuit_id', how='left')

# Add train/test split column
final_df['split'] = final_df['season'].apply(lambda s: 'test' if s == 2025 else 'train')

# Drop rows with null target (can't train/evaluate on them)
before = len(final_df)
final_df = final_df[final_df['position_gain'].notna()].copy()
print(f'Dropped {before - len(final_df)} rows with null position_gain')

print(f'\n✅ Final dataset: {final_df.shape[0]:,} rows × {final_df.shape[1]} cols')
print(f'   Train: {(final_df["split"]=="train").sum():,}')
print(f'   Test:  {(final_df["split"]=="test").sum():,}')

final_df.to_csv(FINAL_DATA_PATH, index=False)
print(f'\n💾 Saved: {FINAL_DATA_PATH}')

## Step 11 — Save Clustering Models

In [ ]:
models_to_save = {
    'kmeans_drivers.pkl':    km_drivers,
    'kmeans_circuits.pkl':   km_circuits,
    'scaler_drivers.pkl':    driver_scaler,
    'scaler_circuits.pkl':   circuit_scaler,
}

for filename, obj in models_to_save.items():
    path = os.path.join(MODELS_DIR, filename)
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    print(f'💾 Saved: models/{filename}')